In [0]:
from pyspark.sql import functions as F

silver_safety_incidents = spark.table(
    "workspace.transportation_analytics.silver_safety_incidents"
)

silver_drivers = spark.table(
    "workspace.transportation_analytics.silver_drivers"
)

silver_trucks = spark.table(
    "workspace.transportation_analytics.silver_trucks"
)

silver_routes = spark.table(
    "workspace.transportation_analytics.silver_routes"
)

print("Safety Metrics Silver tables loaded successfully")

Safety Metrics Silver tables loaded successfully


In [0]:
silver_safety_incidents.printSchema()
silver_trucks.printSchema()
silver_routes.printSchema()
silver_drivers.printSchema()

root
 |-- incident_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- incident_date: timestamp (nullable = true)
 |-- incident_type: string (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)
 |-- at_fault_flag: boolean (nullable = true)
 |-- injury_flag: boolean (nullable = true)
 |-- vehicle_damage_cost: double (nullable = true)
 |-- cargo_damage_cost: double (nullable = true)
 |-- claim_amount: double (nullable = true)
 |-- preventable_flag: boolean (nullable = true)
 |-- description: string (nullable = true)

root
 |-- truck_id: string (nullable = true)
 |-- unit_number: integer (nullable = true)
 |-- make: string (nullable = true)
 |-- model_year: integer (nullable = true)
 |-- vin: string (nullable = true)
 |-- acquisition_date: date (nullable = true)
 |-- acquisition_mileage: integer (nullable = true)
 |-- fuel_type: stri

In [0]:
gold_safety_metrics = (
    silver_safety_incidents
    .groupBy("driver_id")
    .agg(
        F.countDistinct("incident_id").alias("total_incidents"),
        F.sum(
            F.when(F.col("preventable_flag") == True, 1).otherwise(0)
        ).alias("preventable_accidents"),
        F.sum(
            F.when(F.col("at_fault_flag") == True, 1).otherwise(0)
        ).alias("at_fault_incidents"),
        F.sum(
            F.when(F.col("injury_flag") == True, 1).otherwise(0)
        ).alias("injury_incidents"),
        F.sum("vehicle_damage_cost").alias("total_vehicle_damage_cost"),
        F.sum("cargo_damage_cost").alias("total_cargo_damage_cost"),
        F.sum("claim_amount").alias("total_claim_amount")
    )
)

display(gold_safety_metrics)

driver_id,total_incidents,preventable_accidents,at_fault_incidents,injury_incidents,total_vehicle_damage_cost,total_cargo_damage_cost,total_claim_amount
DRV00133,1,0,0,0,14951.4,0.0,14951.4
DRV00101,3,2,3,0,30491.43,31507.48,61998.91
DRV00030,1,0,1,0,16566.78,0.0,16566.78
DRV00003,6,2,1,2,42095.75,6604.51,48700.259999999995
null,1,1,0,0,0.0,0.0,0.0
DRV00122,2,1,0,0,21025.27,9329.09,30354.36
DRV00095,2,1,0,0,11464.33,0.0,11464.33
DRV00016,1,0,0,1,19126.05,0.0,19126.05
DRV00118,1,0,0,0,19578.75,18583.37,38162.12
DRV00054,2,1,1,0,24239.68,0.0,24239.68


In [0]:
gold_safety_metrics = (
    gold_safety_metrics
    .join(
        silver_drivers.select(
            "driver_id",
            "first_name",
            "last_name",
            "employment_status"
        ),
        on="driver_id",
        how="left"
    )
)

display(gold_safety_metrics)

driver_id,total_incidents,preventable_accidents,at_fault_incidents,injury_incidents,total_vehicle_damage_cost,total_cargo_damage_cost,total_claim_amount,first_name,last_name,employment_status
DRV00003,6,2,1,2,42095.75,6604.51,48700.26000000001,Charles,Hernandez,Active
DRV00011,2,1,0,0,40207.43,8080.5,48287.93,Michael,Martinez,Active
DRV00074,1,0,0,0,17565.57,0.0,17565.57,William,Brown,Active
DRV00124,2,1,2,1,41150.33,0.0,41150.33,Thomas,Jackson,Active
DRV00094,2,0,1,0,28429.949999999997,60483.649999999994,88913.6,Mary,Taylor,Active
DRV00018,3,1,0,0,27601.84,408.98,28010.82,Patricia,Martin,Active
DRV00107,1,1,0,1,12833.93,0.0,12833.93,Charles,Jones,Active
DRV00079,1,0,1,0,3082.44,0.0,3082.44,Patricia,Moore,Active
DRV00147,2,1,1,0,8811.67,41316.05,50127.72,Richard,Garcia,Active
DRV00062,1,0,0,0,3931.12,40113.96,44045.08,Jessica,Davis,Active


In [0]:
silver_trips = spark.table(
    "workspace.transportation_analytics.silver_trips"
)

silver_loads = spark.table(
    "workspace.transportation_analytics.silver_loads"
)

print("Trips and Loads tables loaded successfully")

Trips and Loads tables loaded successfully


In [0]:
incident_route = (
    silver_safety_incidents
    .join(
        silver_trips.select(
            "trip_id",
            "load_id"
        ),
        on="trip_id",
        how="left"
    )
    .join(
        silver_loads.select(
            "load_id",
            "route_id"
        ),
        on="load_id",
        how="left"
    )
)

display(
    incident_route.select(
        "incident_id",
        "driver_id",
        "truck_id",
        "incident_date",
        "incident_type",
        "preventable_flag",
        "at_fault_flag",
        "route_id"
    )
)

incident_id,driver_id,truck_id,incident_date,incident_type,preventable_flag,at_fault_flag,route_id
INC00000005,DRV00133,TRK00014,2024-03-20T14:00:00.000Z,DOT Violation,false,false,RTE00021
INC00000060,DRV00101,null,2024-05-21T07:00:00.000Z,Accident,false,true,RTE00017
INC00000068,DRV00030,TRK00092,2024-12-02T08:00:00.000Z,Moving Violation,false,true,RTE00004
INC00000075,DRV00003,TRK00050,2023-01-11T08:00:00.000Z,DOT Violation,false,true,RTE00056
INC00000086,null,TRK00120,2024-10-10T15:00:00.000Z,DOT Violation,true,false,RTE00058
INC00000098,DRV00122,TRK00060,2024-05-07T13:00:00.000Z,Customer Complaint,true,false,RTE00001
INC00000118,DRV00095,TRK00065,2022-05-04T02:00:00.000Z,DOT Violation,true,false,RTE00043
INC00000119,DRV00016,TRK00036,2024-11-14T11:00:00.000Z,Customer Complaint,false,false,RTE00029
INC00000134,DRV00118,TRK00010,2024-08-08T22:00:00.000Z,Equipment Damage,false,false,RTE00057
INC00000138,DRV00054,TRK00098,2023-04-21T08:00:00.000Z,Customer Complaint,false,false,RTE00021


In [0]:
gold_safety_metrics = (
    incident_route
    .groupBy(
        "driver_id",
        "truck_id",
        "route_id"
    )
    .agg(
        F.countDistinct("incident_id").alias("total_incidents"),
        F.sum(
            F.when(F.col("preventable_flag") == True, 1).otherwise(0)
        ).alias("preventable_accidents"),
        F.sum(
            F.when(F.col("at_fault_flag") == True, 1).otherwise(0)
        ).alias("at_fault_incidents"),
        F.sum(
            F.when(F.col("injury_flag") == True, 1).otherwise(0)
        ).alias("injury_incidents"),
        F.sum("vehicle_damage_cost").alias("total_vehicle_damage_cost"),
        F.sum("cargo_damage_cost").alias("total_cargo_damage_cost"),
        F.sum("claim_amount").alias("total_claim_amount")
    )
)

display(gold_safety_metrics)

driver_id,truck_id,route_id,total_incidents,preventable_accidents,at_fault_incidents,injury_incidents,total_vehicle_damage_cost,total_cargo_damage_cost,total_claim_amount
DRV00124,TRK00093,RTE00009,1,0,1,1,23366.57,0.0,23366.57
DRV00102,TRK00036,RTE00028,1,0,0,1,18614.97,0.0,18614.97
DRV00095,TRK00048,RTE00034,1,0,0,0,7869.76,0.0,7869.76
DRV00056,TRK00038,RTE00046,1,0,0,0,0.0,18410.14,18410.14
DRV00037,TRK00040,RTE00021,1,0,0,0,0.0,0.0,0.0
DRV00058,TRK00097,RTE00011,1,0,0,0,15928.79,0.0,15928.79
DRV00099,TRK00025,RTE00039,1,0,0,0,12831.58,0.0,12831.58
DRV00104,TRK00040,RTE00012,1,1,0,1,0.0,0.0,0.0
DRV00018,TRK00112,RTE00010,1,1,0,0,13425.42,376.13,13801.55
DRV00030,TRK00092,RTE00004,1,0,1,0,16566.78,0.0,16566.78


In [0]:
gold_safety_metrics = (
    gold_safety_metrics
    .join(
        silver_drivers.select(
            "driver_id",
            "first_name",
            "last_name",
            "employment_status"
        ),
        on="driver_id",
        how="left"
    )
    .join(
        silver_trucks.select(
            "truck_id",
            "unit_number",
            "make",
            "model_year",
            "status"
        ),
        on="truck_id",
        how="left"
    )
)

display(gold_safety_metrics)

truck_id,driver_id,route_id,total_incidents,preventable_accidents,at_fault_incidents,injury_incidents,total_vehicle_damage_cost,total_cargo_damage_cost,total_claim_amount,first_name,last_name,employment_status,unit_number,make,model_year,status
TRK00093,DRV00124,RTE00009,1,0,1,1,23366.57,0.0,23366.57,Thomas,Jackson,Active,7126,Peterbilt,2015,Active
TRK00036,DRV00102,RTE00028,1,0,0,1,18614.97,0.0,18614.97,Elizabeth,Rodriguez,Active,1618,Peterbilt,2020,Active
TRK00048,DRV00095,RTE00034,1,0,0,0,7869.76,0.0,7869.76,Linda,Thomas,Active,4891,Volvo,2015,Active
TRK00038,DRV00056,RTE00046,1,0,0,0,0.0,18410.14,18410.14,William,Wilson,Active,3147,International,2015,Active
TRK00040,DRV00037,RTE00021,1,0,0,0,0.0,0.0,0.0,Jessica,Garcia,Active,3527,International,2015,Active
TRK00097,DRV00058,RTE00011,1,0,0,0,15928.79,0.0,15928.79,David,Miller,Active,7142,Peterbilt,2016,Active
TRK00025,DRV00099,RTE00039,1,0,0,0,12831.58,0.0,12831.58,Michael,Williams,Active,8967,Freightliner,2018,Active
TRK00040,DRV00104,RTE00012,1,1,0,1,0.0,0.0,0.0,Richard,Hernandez,Active,3527,International,2015,Active
TRK00112,DRV00018,RTE00010,1,1,0,0,13425.42,376.13,13801.55,Patricia,Martin,Active,2298,Volvo,2016,Active
TRK00092,DRV00030,RTE00004,1,0,1,0,16566.78,0.0,16566.78,Jessica,Hernandez,Active,9224,Volvo,2015,Active


In [0]:
display(
    gold_safety_metrics.select(
        "driver_id",
        "first_name",
        "last_name",
        "truck_id",
        "unit_number",
        "route_id",
        "total_incidents",
        "preventable_accidents",
        "at_fault_incidents",
        "injury_incidents",
        "total_vehicle_damage_cost",
        "total_cargo_damage_cost",
        "total_claim_amount"
    )
)

driver_id,first_name,last_name,truck_id,unit_number,route_id,total_incidents,preventable_accidents,at_fault_incidents,injury_incidents,total_vehicle_damage_cost,total_cargo_damage_cost,total_claim_amount
DRV00124,Thomas,Jackson,TRK00093,7126,RTE00009,1,0,1,1,23366.57,0.0,23366.57
DRV00102,Elizabeth,Rodriguez,TRK00036,1618,RTE00028,1,0,0,1,18614.97,0.0,18614.97
DRV00095,Linda,Thomas,TRK00048,4891,RTE00034,1,0,0,0,7869.76,0.0,7869.76
DRV00056,William,Wilson,TRK00038,3147,RTE00046,1,0,0,0,0.0,18410.14,18410.14
DRV00037,Jessica,Garcia,TRK00040,3527,RTE00021,1,0,0,0,0.0,0.0,0.0
DRV00058,David,Miller,TRK00097,7142,RTE00011,1,0,0,0,15928.79,0.0,15928.79
DRV00099,Michael,Williams,TRK00025,8967,RTE00039,1,0,0,0,12831.58,0.0,12831.58
DRV00104,Richard,Hernandez,TRK00040,3527,RTE00012,1,1,0,1,0.0,0.0,0.0
DRV00018,Patricia,Martin,TRK00112,2298,RTE00010,1,1,0,0,13425.42,376.13,13801.55
DRV00030,Jessica,Hernandez,TRK00092,9224,RTE00004,1,0,1,0,16566.78,0.0,16566.78


In [0]:
gold_safety_metrics.limit(0).write \
    .format("delta") \
    .saveAsTable(
        "workspace.transportation_analytics.gold_safety_metrics"
    )

print("Empty Gold Safety Metrics table created successfully")

Empty Gold Safety Metrics table created successfully


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_safety_metrics"
)

target.alias("t").merge(
    gold_safety_metrics.alias("s"),
    """
    t.driver_id = s.driver_id
    AND t.truck_id = s.truck_id
    AND t.route_id = s.route_id
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Safety Metrics table updated using MERGE")

Gold Safety Metrics table updated using MERGE


In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_safety_metrics"
    )
)

truck_id,driver_id,route_id,total_incidents,preventable_accidents,at_fault_incidents,injury_incidents,total_vehicle_damage_cost,total_cargo_damage_cost,total_claim_amount,first_name,last_name,employment_status,unit_number,make,model_year,status
TRK00093,DRV00124,RTE00009,1,0,1,1,23366.57,0.0,23366.57,Thomas,Jackson,Active,7126,Peterbilt,2015,Active
TRK00036,DRV00102,RTE00028,1,0,0,1,18614.97,0.0,18614.97,Elizabeth,Rodriguez,Active,1618,Peterbilt,2020,Active
TRK00048,DRV00095,RTE00034,1,0,0,0,7869.76,0.0,7869.76,Linda,Thomas,Active,4891,Volvo,2015,Active
TRK00038,DRV00056,RTE00046,1,0,0,0,0.0,18410.14,18410.14,William,Wilson,Active,3147,International,2015,Active
TRK00040,DRV00037,RTE00021,1,0,0,0,0.0,0.0,0.0,Jessica,Garcia,Active,3527,International,2015,Active
TRK00097,DRV00058,RTE00011,1,0,0,0,15928.79,0.0,15928.79,David,Miller,Active,7142,Peterbilt,2016,Active
TRK00025,DRV00099,RTE00039,1,0,0,0,12831.58,0.0,12831.58,Michael,Williams,Active,8967,Freightliner,2018,Active
TRK00040,DRV00104,RTE00012,1,1,0,1,0.0,0.0,0.0,Richard,Hernandez,Active,3527,International,2015,Active
TRK00112,DRV00018,RTE00010,1,1,0,0,13425.42,376.13,13801.55,Patricia,Martin,Active,2298,Volvo,2016,Active
TRK00092,DRV00030,RTE00004,1,0,1,0,16566.78,0.0,16566.78,Jessica,Hernandez,Active,9224,Volvo,2015,Active


In [0]:
display(
    gold_safety_metrics
    .groupBy(
        "driver_id",
        "first_name",
        "last_name"
    )
    .agg(
        F.sum("preventable_accidents").alias(
            "total_preventable_accidents"
        )
    )
    .orderBy(F.desc("total_preventable_accidents"))
)

driver_id,first_name,last_name,total_preventable_accidents
DRV00098,David,Anderson,3
DRV00058,David,Miller,3
DRV00104,Richard,Hernandez,3
DRV00070,Sarah,Anderson,3
DRV00012,Robert,Moore,2
DRV00033,William,Anderson,2
DRV00003,Charles,Hernandez,2
DRV00066,William,Wilson,2
DRV00013,Jessica,Jones,2
DRV00119,Jennifer,Wilson,2


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_safety_metrics
    .groupBy("truck_id")
    .agg(
        F.sum("total_incidents").alias("total_incidents")
    )
    .orderBy(F.desc("total_incidents"))
)

truck_id,total_incidents
TRK00019,6
TRK00007,6
TRK00067,5
TRK00061,5
TRK00031,5
TRK00069,4
TRK00054,4
TRK00062,4
TRK00024,4
TRK00057,4


Databricks visualization. Run in Databricks to view.

In [0]:
gold_safety_metrics = (
    incident_route
    .withColumn(
        "incident_month",
        F.date_trunc("month", F.col("incident_date"))
    )
    .groupBy(
        "driver_id",
        "truck_id",
        "route_id",
        "incident_month"
    )
    .agg(
        F.countDistinct("incident_id").alias("total_incidents"),
        F.sum(
            F.when(F.col("preventable_flag") == True, 1).otherwise(0)
        ).alias("preventable_accidents"),
        F.sum(
            F.when(F.col("at_fault_flag") == True, 1).otherwise(0)
        ).alias("at_fault_incidents"),
        F.sum(
            F.when(F.col("injury_flag") == True, 1).otherwise(0)
        ).alias("injury_incidents"),
        F.sum("vehicle_damage_cost").alias("total_vehicle_damage_cost"),
        F.sum("cargo_damage_cost").alias("total_cargo_damage_cost"),
        F.sum("claim_amount").alias("total_claim_amount")
    )
)

display(gold_safety_metrics)

driver_id,truck_id,route_id,incident_month,total_incidents,preventable_accidents,at_fault_incidents,injury_incidents,total_vehicle_damage_cost,total_cargo_damage_cost,total_claim_amount
DRV00132,TRK00026,RTE00039,2024-04-01T00:00:00.000Z,1,0,1,0,13452.82,0.0,13452.82
DRV00050,TRK00092,RTE00013,2022-10-01T00:00:00.000Z,1,0,0,1,0.0,0.0,0.0
DRV00146,TRK00083,RTE00045,2023-05-01T00:00:00.000Z,1,0,0,1,18884.39,29755.82,48640.21
DRV00104,TRK00013,RTE00029,2022-06-01T00:00:00.000Z,1,1,1,0,13379.62,2259.73,15639.35
DRV00006,TRK00084,RTE00040,2023-02-01T00:00:00.000Z,1,0,0,0,2700.7,14284.24,16984.94
DRV00079,TRK00099,RTE00048,2023-03-01T00:00:00.000Z,1,0,1,0,3082.44,0.0,3082.44
DRV00012,TRK00061,RTE00057,2022-08-01T00:00:00.000Z,1,1,0,0,15647.86,2310.73,17958.59
DRV00006,TRK00006,RTE00042,2023-04-01T00:00:00.000Z,1,1,1,0,12629.26,0.0,12629.26
DRV00050,TRK00019,RTE00005,2022-01-01T00:00:00.000Z,1,1,1,0,11269.86,0.0,11269.86
DRV00088,TRK00020,RTE00019,2024-05-01T00:00:00.000Z,1,0,1,0,24614.16,0.0,24614.16


In [0]:
spark.sql("""
ALTER TABLE workspace.transportation_analytics.gold_safety_metrics
ADD COLUMNS (
    incident_month TIMESTAMP
)
""")

print("incident_month column added successfully")

incident_month column added successfully


In [0]:
gold_safety_metrics = (
    gold_safety_metrics
    .join(
        silver_drivers.select(
            "driver_id",
            "first_name",
            "last_name",
            "employment_status"
        ),
        on="driver_id",
        how="left"
    )
    .join(
        silver_trucks.select(
            "truck_id",
            "unit_number",
            "make",
            "model_year",
            "status"
        ),
        on="truck_id",
        how="left"
    )
)

display(gold_safety_metrics)

truck_id,driver_id,route_id,incident_month,total_incidents,preventable_accidents,at_fault_incidents,injury_incidents,total_vehicle_damage_cost,total_cargo_damage_cost,total_claim_amount,first_name,last_name,employment_status,unit_number,make,model_year,status
TRK00026,DRV00132,RTE00039,2024-04-01T00:00:00.000Z,1,0,1,0,13452.82,0.0,13452.82,Patricia,Gonzalez,Active,4145,International,2021,Active
TRK00092,DRV00050,RTE00013,2022-10-01T00:00:00.000Z,1,0,0,1,0.0,0.0,0.0,Robert,Johnson,Active,9224,Volvo,2015,Active
TRK00083,DRV00146,RTE00045,2023-05-01T00:00:00.000Z,1,0,0,1,18884.39,29755.82,48640.21,Jessica,Gonzalez,Active,8167,Kenworth,2015,Active
TRK00013,DRV00104,RTE00029,2022-06-01T00:00:00.000Z,1,1,1,0,13379.62,2259.73,15639.35,Richard,Hernandez,Active,3093,Volvo,2018,Active
TRK00084,DRV00006,RTE00040,2023-02-01T00:00:00.000Z,1,0,0,0,2700.7,14284.24,16984.94,Robert,Taylor,Active,2502,Freightliner,2015,Active
TRK00099,DRV00079,RTE00048,2023-03-01T00:00:00.000Z,1,0,1,0,3082.44,0.0,3082.44,Patricia,Moore,Active,1461,Freightliner,2015,Active
TRK00061,DRV00012,RTE00057,2022-08-01T00:00:00.000Z,1,1,0,0,15647.86,2310.73,17958.59,Robert,Moore,Active,5616,International,2018,Active
TRK00006,DRV00006,RTE00042,2023-04-01T00:00:00.000Z,1,1,1,0,12629.26,0.0,12629.26,Robert,Taylor,Active,6082,Kenworth,2017,Active
TRK00019,DRV00050,RTE00005,2022-01-01T00:00:00.000Z,1,1,1,0,11269.86,0.0,11269.86,Robert,Johnson,Active,1298,International,2015,Active
TRK00020,DRV00088,RTE00019,2024-05-01T00:00:00.000Z,1,0,1,0,24614.16,0.0,24614.16,Joseph,Garcia,Active,6940,Kenworth,2015,Active


In [0]:
target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_safety_metrics"
)

target.alias("t").merge(
    gold_safety_metrics.alias("s"),
    """
    t.driver_id = s.driver_id
    AND t.truck_id = s.truck_id
    AND t.route_id = s.route_id
    AND t.incident_month = s.incident_month
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Safety Metrics table updated successfully with time period")

Gold Safety Metrics table updated successfully with time period


In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_safety_metrics"
    )
)

truck_id,driver_id,route_id,total_incidents,preventable_accidents,at_fault_incidents,injury_incidents,total_vehicle_damage_cost,total_cargo_damage_cost,total_claim_amount,first_name,last_name,employment_status,unit_number,make,model_year,status,incident_month
TRK00093,DRV00124,RTE00009,1,0,1,1,23366.57,0.0,23366.57,Thomas,Jackson,Active,7126,Peterbilt,2015,Active,null
TRK00036,DRV00102,RTE00028,1,0,0,1,18614.97,0.0,18614.97,Elizabeth,Rodriguez,Active,1618,Peterbilt,2020,Active,null
TRK00048,DRV00095,RTE00034,1,0,0,0,7869.76,0.0,7869.76,Linda,Thomas,Active,4891,Volvo,2015,Active,null
TRK00038,DRV00056,RTE00046,1,0,0,0,0.0,18410.14,18410.14,William,Wilson,Active,3147,International,2015,Active,null
TRK00040,DRV00037,RTE00021,1,0,0,0,0.0,0.0,0.0,Jessica,Garcia,Active,3527,International,2015,Active,null
TRK00097,DRV00058,RTE00011,1,0,0,0,15928.79,0.0,15928.79,David,Miller,Active,7142,Peterbilt,2016,Active,null
TRK00025,DRV00099,RTE00039,1,0,0,0,12831.58,0.0,12831.58,Michael,Williams,Active,8967,Freightliner,2018,Active,null
TRK00040,DRV00104,RTE00012,1,1,0,1,0.0,0.0,0.0,Richard,Hernandez,Active,3527,International,2015,Active,null
TRK00112,DRV00018,RTE00010,1,1,0,0,13425.42,376.13,13801.55,Patricia,Martin,Active,2298,Volvo,2016,Active,null
TRK00092,DRV00030,RTE00004,1,0,1,0,16566.78,0.0,16566.78,Jessica,Hernandez,Active,9224,Volvo,2015,Active,null


In [0]:
trip_counts = (
    silver_trips
    .withColumn(
        "trip_month",
        F.date_trunc("month", F.col("dispatch_date"))
    )
    .groupBy(
        "driver_id",
        "truck_id",
        "load_id",
        "trip_month"
    )
    .agg(
        F.countDistinct("trip_id").alias("total_trips")
    )
)

display(trip_counts)

driver_id,truck_id,load_id,trip_month,total_trips
DRV00088,TRK00039,LOAD00000012,2022-01-01T00:00:00.000Z,1
DRV00044,TRK00001,LOAD00000040,2022-01-01T00:00:00.000Z,1
DRV00148,TRK00050,LOAD00000050,2022-01-01T00:00:00.000Z,1
DRV00114,TRK00030,LOAD00000114,2022-01-01T00:00:00.000Z,1
DRV00105,TRK00030,LOAD00000131,2022-01-01T00:00:00.000Z,1
DRV00122,TRK00080,LOAD00000154,2022-01-01T00:00:00.000Z,1
DRV00090,null,LOAD00000159,2022-01-01T00:00:00.000Z,1
DRV00019,TRK00013,LOAD00000180,2022-01-01T00:00:00.000Z,1
DRV00145,TRK00086,LOAD00000181,2022-01-01T00:00:00.000Z,1
DRV00091,TRK00053,LOAD00000192,2022-01-01T00:00:00.000Z,1


In [0]:
gold_safety_metrics = (
    gold_safety_metrics
    .join(
        trip_counts,
        (
            (gold_safety_metrics.driver_id == trip_counts.driver_id) &
            (gold_safety_metrics.truck_id == trip_counts.truck_id) &
            (gold_safety_metrics.incident_month == trip_counts.trip_month)
        ),
        how="left"
    )
    .drop(trip_counts.driver_id)
    .drop(trip_counts.truck_id)
    .drop("trip_month")
)

gold_safety_metrics = (
    gold_safety_metrics
    .withColumn(
        "incident_rate_per_1000_trips",
        F.round(
            F.col("total_incidents") /
            F.col("total_trips") * 1000,
            2
        )
    )
)

display(gold_safety_metrics)

truck_id,driver_id,route_id,incident_month,total_incidents,preventable_accidents,at_fault_incidents,injury_incidents,total_vehicle_damage_cost,total_cargo_damage_cost,total_claim_amount,first_name,last_name,employment_status,unit_number,make,model_year,status,load_id,total_trips,incident_rate_per_1000_trips
TRK00026,DRV00132,RTE00039,2024-04-01T00:00:00.000Z,1,0,1,0,13452.82,0.0,13452.82,Patricia,Gonzalez,Active,4145,International,2021,Active,null,null,null
TRK00092,DRV00050,RTE00013,2022-10-01T00:00:00.000Z,1,0,0,1,0.0,0.0,0.0,Robert,Johnson,Active,9224,Volvo,2015,Active,LOAD00022838,1,1000.0
TRK00083,DRV00146,RTE00045,2023-05-01T00:00:00.000Z,1,0,0,1,18884.39,29755.82,48640.21,Jessica,Gonzalez,Active,8167,Kenworth,2015,Active,LOAD00039355,1,1000.0
TRK00013,DRV00104,RTE00029,2022-06-01T00:00:00.000Z,1,1,1,0,13379.62,2259.73,15639.35,Richard,Hernandez,Active,3093,Volvo,2018,Active,LOAD00012626,1,1000.0
TRK00084,DRV00006,RTE00040,2023-02-01T00:00:00.000Z,1,0,0,0,2700.7,14284.24,16984.94,Robert,Taylor,Active,2502,Freightliner,2015,Active,LOAD00032462,1,1000.0
TRK00099,DRV00079,RTE00048,2023-03-01T00:00:00.000Z,1,0,1,0,3082.44,0.0,3082.44,Patricia,Moore,Active,1461,Freightliner,2015,Active,LOAD00033560,1,1000.0
TRK00061,DRV00012,RTE00057,2022-08-01T00:00:00.000Z,1,1,0,0,15647.86,2310.73,17958.59,Robert,Moore,Active,5616,International,2018,Active,LOAD00018371,1,1000.0
TRK00006,DRV00006,RTE00042,2023-04-01T00:00:00.000Z,1,1,1,0,12629.26,0.0,12629.26,Robert,Taylor,Active,6082,Kenworth,2017,Active,LOAD00036079,1,1000.0
TRK00019,DRV00050,RTE00005,2022-01-01T00:00:00.000Z,1,1,1,0,11269.86,0.0,11269.86,Robert,Johnson,Active,1298,International,2015,Active,LOAD00001736,1,1000.0
TRK00020,DRV00088,RTE00019,2024-05-01T00:00:00.000Z,1,0,1,0,24614.16,0.0,24614.16,Joseph,Garcia,Active,6940,Kenworth,2015,Active,LOAD00068238,1,1000.0


In [0]:
trip_counts = (
    silver_trips
    .withColumn(
        "trip_month",
        F.date_trunc("month", "dispatch_date")
    )
    .groupBy(
        "driver_id",
        "truck_id",
        "trip_month"
    )
    .agg(
        F.countDistinct("trip_id").alias("total_trips")
    )
)

display(trip_counts)

driver_id,truck_id,trip_month,total_trips
DRV00088,TRK00039,2022-01-01T00:00:00.000Z,2
DRV00044,TRK00001,2022-01-01T00:00:00.000Z,1
DRV00148,TRK00050,2022-01-01T00:00:00.000Z,3
DRV00114,TRK00030,2022-01-01T00:00:00.000Z,1
DRV00105,TRK00030,2022-01-01T00:00:00.000Z,1
DRV00122,TRK00080,2022-01-01T00:00:00.000Z,1
DRV00090,null,2022-01-01T00:00:00.000Z,1
DRV00019,TRK00013,2022-01-01T00:00:00.000Z,1
DRV00145,TRK00086,2022-01-01T00:00:00.000Z,1
DRV00091,TRK00053,2022-01-01T00:00:00.000Z,1


In [0]:
gold_safety_metrics = gold_safety_metrics.join(
    trip_counts,
    ["driver_id", "truck_id"],
    "left"
)

In [0]:
gold_safety_metrics = gold_safety_metrics.drop(trip_counts.total_trips)

In [0]:
gold_safety_metrics = spark.table(
    "workspace.transportation_analytics.gold_safety_metrics"
)

display(gold_safety_metrics)

truck_id,driver_id,route_id,total_incidents,preventable_accidents,at_fault_incidents,injury_incidents,total_vehicle_damage_cost,total_cargo_damage_cost,total_claim_amount,first_name,last_name,employment_status,unit_number,make,model_year,status,incident_month,total_trips,incident_rate_per_1000_trips
TRK00093,DRV00124,RTE00009,1,0,1,1,23366.57,0.0,23366.57,Thomas,Jackson,Active,7126,Peterbilt,2015,Active,null,null,null
TRK00036,DRV00102,RTE00028,1,0,0,1,18614.97,0.0,18614.97,Elizabeth,Rodriguez,Active,1618,Peterbilt,2020,Active,null,null,null
TRK00048,DRV00095,RTE00034,1,0,0,0,7869.76,0.0,7869.76,Linda,Thomas,Active,4891,Volvo,2015,Active,null,null,null
TRK00038,DRV00056,RTE00046,1,0,0,0,0.0,18410.14,18410.14,William,Wilson,Active,3147,International,2015,Active,null,null,null
TRK00040,DRV00037,RTE00021,1,0,0,0,0.0,0.0,0.0,Jessica,Garcia,Active,3527,International,2015,Active,null,null,null
TRK00097,DRV00058,RTE00011,1,0,0,0,15928.79,0.0,15928.79,David,Miller,Active,7142,Peterbilt,2016,Active,null,null,null
TRK00025,DRV00099,RTE00039,1,0,0,0,12831.58,0.0,12831.58,Michael,Williams,Active,8967,Freightliner,2018,Active,null,null,null
TRK00040,DRV00104,RTE00012,1,1,0,1,0.0,0.0,0.0,Richard,Hernandez,Active,3527,International,2015,Active,null,null,null
TRK00112,DRV00018,RTE00010,1,1,0,0,13425.42,376.13,13801.55,Patricia,Martin,Active,2298,Volvo,2016,Active,null,null,null
TRK00092,DRV00030,RTE00004,1,0,1,0,16566.78,0.0,16566.78,Jessica,Hernandez,Active,9224,Volvo,2015,Active,null,null,null


In [0]:
display(
    gold_safety_metrics.select(
        "driver_id",
        "truck_id",
        "route_id",
        "incident_month",
        "total_incidents",
        "total_trips",
        "incident_rate_per_1000_trips"
    )
)

driver_id,truck_id,route_id,incident_month,total_incidents,total_trips,incident_rate_per_1000_trips
DRV00124,TRK00093,RTE00009,null,1,null,null
DRV00102,TRK00036,RTE00028,null,1,null,null
DRV00095,TRK00048,RTE00034,null,1,null,null
DRV00056,TRK00038,RTE00046,null,1,null,null
DRV00037,TRK00040,RTE00021,null,1,null,null
DRV00058,TRK00097,RTE00011,null,1,null,null
DRV00099,TRK00025,RTE00039,null,1,null,null
DRV00104,TRK00040,RTE00012,null,1,null,null
DRV00018,TRK00112,RTE00010,null,1,null,null
DRV00030,TRK00092,RTE00004,null,1,null,null


In [0]:
display(
    gold_safety_metrics
    .groupBy("incident_month")
    .agg(
        F.sum("total_incidents").alias("total_incidents"),
        F.sum("preventable_accidents").alias("preventable_accidents")
    )
    .orderBy("incident_month")
)

incident_month,total_incidents,preventable_accidents
null,340,128
2022-01-01T00:00:00.000Z,7,1
2022-02-01T00:00:00.000Z,2,0
2022-03-01T00:00:00.000Z,3,1
2022-04-01T00:00:00.000Z,4,0
2022-05-01T00:00:00.000Z,3,3
2022-06-01T00:00:00.000Z,7,4
2022-07-01T00:00:00.000Z,6,3
2022-08-01T00:00:00.000Z,9,5
2022-09-01T00:00:00.000Z,5,0


In [0]:
display(
    gold_safety_metrics
    .groupBy("driver_id")
    .agg(
        F.sum("total_incidents").alias("total_incidents"),
        F.sum("total_trips").alias("total_trips"),
        F.round(
            F.sum("total_incidents") /
            F.sum("total_trips") * 1000,
            2
        ).alias("incident_rate_per_1000_trips")
    )
    .orderBy(F.desc("incident_rate_per_1000_trips"))
)

driver_id,total_incidents,total_trips,incident_rate_per_1000_trips
DRV00134,6,1,6000.0
DRV00127,6,1,6000.0
DRV00101,11,2,5500.0
DRV00132,9,2,4500.0
DRV00003,18,5,3600.0
DRV00045,3,1,3000.0
DRV00044,6,2,3000.0
DRV00002,6,2,3000.0
DRV00028,3,1,3000.0
DRV00017,3,1,3000.0


In [0]:
display(
    gold_safety_metrics
    .groupBy("truck_id")
    .agg(
        F.sum("total_incidents").alias("total_incidents"),
        F.sum("total_trips").alias("total_trips"),
        F.round(
            F.sum("total_incidents") /
            F.sum("total_trips") * 1000,
            2
        ).alias("incident_rate_per_1000_trips")
    )
    .orderBy(F.desc("incident_rate_per_1000_trips"))
)

truck_id,total_incidents,total_trips,incident_rate_per_1000_trips
TRK00120,8,1,8000.0
TRK00073,6,1,6000.0
TRK00110,6,1,6000.0
TRK00026,9,2,4500.0
TRK00004,6,2,3000.0
TRK00083,6,2,3000.0
TRK00089,3,1,3000.0
TRK00035,3,1,3000.0
TRK00018,3,1,3000.0
TRK00007,18,6,3000.0


In [0]:
display(
    gold_safety_metrics
    .groupBy("incident_month")
    .agg(
        F.sum("total_incidents").alias("total_incidents"),
        F.sum("preventable_accidents").alias("preventable_accidents")
    )
    .orderBy("incident_month")
)

incident_month,total_incidents,preventable_accidents
null,340,128
2022-01-01T00:00:00.000Z,7,1
2022-02-01T00:00:00.000Z,2,0
2022-03-01T00:00:00.000Z,3,1
2022-04-01T00:00:00.000Z,4,0
2022-05-01T00:00:00.000Z,3,3
2022-06-01T00:00:00.000Z,7,4
2022-07-01T00:00:00.000Z,6,3
2022-08-01T00:00:00.000Z,9,5
2022-09-01T00:00:00.000Z,5,0


Databricks visualization. Run in Databricks to view.